In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

In [ ]:
import selenium
print(selenium.__version__)

4.44.0


In [ ]:
def ls_to_eu(ls):
    ls = str(ls).replace("£", "") # retirer la devise
    return float(ls) * 1.16 # convertit livre en euro


Étape 1 — parcourt les pages uniquement pour collecter les URLs

Étape 2 — visite chaque URL directement, les pages n'ont plus aucune importance

In [ ]:
driver = webdriver.Chrome()
driver.get("https://books.toscrape.com/")
data = []
wait = WebDriverWait(driver, 3) # attend 3 secondes si un élément est absent

# Liste vide pour la collecte de urls
book_urls = []

# ÉTAPE 1 : Collecter toutes les URLs
while True: # boucle infinie, on sort manuellement avec break

    # Attend que les liens soient présents dans le DOM
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.product_pod h3 a")))

    # Trouve tous les liens et extrait leur href, les ajoute à book_urls
    book_urls += [link.get_attribute("href") for link in driver.find_elements(By.CSS_SELECTOR, "article.product_pod h3 a")]

    try: # cherche le bouton "next" clique dessus directement si trouvé
        wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a"))).click()
    except:
        # bouton "next" absent = dernière page on sort de la boucle
        break

print(f"{len(book_urls)} URLs collectées")

# ÉTAPE 2 : Extraire les données de chaque livre
SELECTORS = {
    "categorie":   (By.XPATH, "(//ul[@class='breadcrumb']//a)[last()]"),
    "titre":       (By.CSS_SELECTOR, "h1"),
    "prix":        (By.CSS_SELECTOR, ".price_color"),
    "stock":       (By.CSS_SELECTOR, ".instock.availability"),
    "description": (By.CSS_SELECTOR, "#product_description + p"),
}

for url in book_urls:
    try:
        driver.get(url) # navigue directement vers la page du livre
        data.append({
                        key: wait.until(EC.presence_of_element_located(sel)).text # attend que l'élément soit présent
                        for key, sel in SELECTORS.items()}) # décompose chaque entrée du dict
    except Exception as e:
        print(f"Erreur sur {url} : {e}")

dff = pd.DataFrame(data)

driver.quit()

print(f"{len(dff)} lignes enregistrées")

1000 URLs collectées
Erreur sur https://books.toscrape.com/catalogue/the-bridge-to-consciousness-im-writing-the-bridge-between-science-and-our-old-and-new-beliefs_840/index.html : Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7e26efd95+15395]
	chromedriver!GetHandleVerifier [0x7ff7e26efdf0+153f0]
	chromedriver!(No symbol) [0x7ff7e22278fd]
	chromedriver!(No symbol) [0x7ff7e2281479]
	chromedriver!(No symbol) [0x7ff7e228177c]
	chromedriver!(No symbol) [0x7ff7e22d1ef7]
	chromedriver!(No symbol) [0x7ff7e22ceaeb]
	chromedriver!(No symbol) [0x7ff7e2273a58]
	chromedriver!(No symbol) [0x7ff7e2274953]
	chromedriver!GetHandleVerifier [0x7ff7e2cb5511+5dab11]
	chromedriver!GetHandleVerifier [0x7ff7e2caf8fb+5d4efb]
	chromedriver!GetHandleVerifier [0x7ff7e2cd3205+5f8805]
	chromedriver!GetHandleVerifier [0x7ff7e270cc6e+3226e]
	chromedriver!GetHandleVerifier [0x7ff7e271562c+3ac2c]
	chromedriver!GetHandleVerifier [0x7ff7e26f9984+1ef84]
	chromedriver!GetHandleVerifier [0x7ff7e26f9b14+1f114]

2 erreurs car pas de description

In [ ]:
dff

,categorie,titre,prix,stock,description
0,Poetry,A Light in the Attic,£51.77,In stock (22 available),It's hard to imagine a world without A Light i...
1,Historical Fiction,Tipping the Velvet,£53.74,In stock (20 available),"""Erotic and absorbing...Written with starling ..."
2,Fiction,Soumission,£50.10,In stock (20 available),"Dans une France assez proche de la nôtre, un h..."
3,Mystery,Sharp Objects,£47.82,In stock (20 available),"WICKED above her hipbone, GIRL across her hear..."
4,History,Sapiens: A Brief History of Humankind,£54.23,In stock (20 available),From a renowned historian comes a groundbreaki...
...,...,...,...,...,...
993,Philosophy,Beyond Good and Evil,£43.38,In stock (1 available),Friedrich Nietzsche's Beyond Good and Evil is ...
994,Sequential Art,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",£57.06,In stock (1 available),High school student Kei Nagai is struck dead i...
995,Historical Fiction,A Spy's Devotion (The Regency Spies of London #1),£16.97,In stock (1 available),"In England’s Regency era, manners and elegance..."
996,Mystery,1st to Die (Women's Murder Club #1),£53.98,In stock (1 available),"James Patterson, bestselling author of the Ale..."


In [ ]:
dff.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 998 entries, 0 to 997
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   categorie    998 non-null    object
 1   titre        998 non-null    object
 2   prix         998 non-null    object
 3   stock        998 non-null    object
 4   description  998 non-null    object
dtypes: object(5)
memory usage: 39.1+ KB


                                                                Nettoyage

In [ ]:
dff["prix"] = round(dff["prix"].apply(ls_to_eu), 2)

In [ ]:
# 1 : split : ["In stock ", "22 available)"]
# 2 : on garde le deuxime element
# 3 : split : ["22", "available)"]
# 4 : on garde le premiere element

dff["stock"] = dff["stock"].apply(lambda text: text.split("(")[1].split()[0])

In [ ]:
dff["stock"] = dff["stock"].astype(int) # converti en entier

In [ ]:
dff.head()

,categorie,titre,prix,stock,description
0,Poetry,A Light in the Attic,60.05,22,It's hard to imagine a world without A Light i...
1,Historical Fiction,Tipping the Velvet,62.34,20,"""Erotic and absorbing...Written with starling ..."
2,Fiction,Soumission,58.12,20,"Dans une France assez proche de la nôtre, un h..."
3,Mystery,Sharp Objects,55.47,20,"WICKED above her hipbone, GIRL across her hear..."
4,History,Sapiens: A Brief History of Humankind,62.91,20,From a renowned historian comes a groundbreaki...


                                                               Exploration

In [ ]:
dff.describe()

,prix,stock
count,998.000000,998.000000
mean,40.661333,8.586172
std,16.757810,5.651540
min,11.600000,1.000000
25%,25.642500,3.000000
50%,41.735000,7.000000
75%,54.992500,14.000000
max,69.590000,22.000000


In [ ]:

dff["categorie"].describe() # verifie le nb de categorie


count         998
unique         50
top       Default
freq          151
Name: categorie, dtype: object

In [ ]:

dff["categorie"].value_counts()


categorie
Default               151
Nonfiction            110
Sequential Art         75
Add a comment          67
Fiction                65
Young Adult            54
Fantasy                48
Romance                35
Mystery                32
Food and Drink         30
Childrens              29
Historical Fiction     26
Poetry                 19
Classics               18
History                18
Horror                 17
Womens Fiction         17
Science Fiction        16
Science                14
Music                  13
Business               12
Thriller               11
Travel                 11
Philosophy             11
Humor                  10
Autobiography           9
Art                     8
Psychology              7
Religion                7
Spirituality            6
Christian Fiction       6
New Adult               6
Sports and Games        5
Biography               5
Self Help               5
Health                  4
Christian               3
Politics                3
Co

In [ ]:

dff["titre"].describe() # verifie les doublons


count                        998
unique                       997
top       The Star-Touched Queen
freq                           2
Name: titre, dtype: object

In [ ]:

dff[dff["titre"] == "The Star-Touched Queen"]


,categorie,titre,prix,stock,description
235,Fantasy,The Star-Touched Queen,53.38,14,Fate and fortune. Power and passion. What does...
357,Fantasy,The Star-Touched Queen,37.47,12,Fate and fortune. Power and passion. What does...


In [ ]:
dff_sorted = dff.sort_values(by="categorie")

dff_sorted.head()

,categorie,titre,prix,stock,description
615,Academic,Logan Kade (Fallen Crest High #5.5),15.22,5,People think that just because they know my na...
922,Add a comment,The Odyssey,34.38,1,Literature's grandest evocation of life's jour...
443,Add a comment,23 Degrees South: A Tropical Tale of Changing ...,41.52,9,"""Consistently entertaining courtesy of Rabin's..."
55,Add a comment,The Torch Is Passed: A Harding Family Story,22.14,16,Andrea Harding is a recent college graduate lo...
231,Add a comment,The White Queen (The Cousins' War #1),30.06,14,Philippa Gregory presents the first of a new s...
